In [1]:
import logging
import warnings
import shutil
import yaml

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import roc_auc_score
import warnings


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
logging.getLogger().setLevel(logging.WARNING)
warnings.filterwarnings('ignore')
sns.set_palette('bright')


pd.options.display.float_format = "{:.2f}".format
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

In [3]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

# Входные данные


## Загрузка данных


In [4]:
path = r'/content/drive/MyDrive/colab/year_project/data/data.pqt'
local_path = r'/content/data.pqt'

shutil.copy2(path, local_path)

'/content/data.pqt'

In [5]:
path = r'/content/drive/MyDrive/colab/year_project/models/params/features.yaml'
with open(path, 'r') as file:
    BEST_FEATURES = yaml.load(file, Loader=yaml.FullLoader)['FINAL_FEATURES']

In [7]:
TOP_100_FEATURES = [
    # Транзакционные признаки
    'TransactionAmt',
    'ProductCD',
    
    # Карточные признаки
    'card2', 'card3', 'card4', 'card5', 'card6',
    
    # Адресные признаки
    'addr1', 'addr2', 'dist1', 'dist2',
    
    # Email домены
    'P_emaildomain', 'R_emaildomain',
    
    # C признаки
    'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10',
    'C11', 'C12', 'C13', 'C14',
    
    # D признаки (временные)
    'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10',
    'D11', 'D12', 'D13', 'D14', 'D15',
    
    # M признаки (M1-M9)
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    
    # V признаки
    'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
    'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
    'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30',
    'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40',
    
    # Временные агрегации
    'hour', 'minute', 'second',
    'is_night',
    
    # Агрегации по сумме
    'amt_mean_168h', 'amt_median_168h', 'amt_std_168h',
    'amt_min_168h', 'amt_max_168h', 'amt_sum_168h',
    'amt_ratio_to_mean_168h', 'amt_ratio_to_median_168h',
    
    # Агрегации по времени
    'count_168h', 'mean_gap_168h', 'min_gap_168h', 'max_gap_168h',
    'time_since_last_168h', 'log_time_since_last_168h',
    
    # Агрегации по карте
    'card_amt_mean', 'card_amt_median', 'card_amt_std',
    'card_amt_max', 'card_amt_min',
    'card_amt_ratio_to_mean', 'card_amt_ratio_to_median',
    'card_amt_more_2x_mean', 'card_amt_more_3x_mean',
    
    # Поведенческие
    'card_unique_P_email', 'card_unique_R_email',
    'card_unique_P_email_gr', 'card_unique_R_email_gr',
    'card_unique_addr1', 'card_unique_addr2',
    'card_mode_hour', 'card_hour_changed',
    'card_night_ratio', 'card_weekend_ratio',
    'card_addr1_changed', 'card_addr2_changed',
]

In [8]:
columns_needed = ['card1', 'TransactionID', 'date_month', 'date_quartal', 'target', 'sample_type', 'competition_sample_type'] + TOP_100_FEATURES

In [9]:
data = pd.read_parquet(local_path, columns=columns_needed)
data.shape

(1097231, 137)

## Валидные переменные


In [10]:
path = r'/content/drive/MyDrive/colab/year_project/docs/valid_features.xlsx'
valid_features_data = pd.read_excel(path, index_col=False)

## Конфиги/константы


In [11]:
TARGET = 'target'
DATE_MONTH = 'date_month'
DATE_QUARTAL = 'date_quartal'

# FEATURES = valid_features_data.loc[(
#     valid_features_data['valid_flag'] == 1)]['attribute'].values
CAT_FEATURES = sorted(set(TOP_100_FEATURES) & set(
    data.select_dtypes(include=["object", "category"]).columns))
NUM_FEATURES = sorted(set(TOP_100_FEATURES) & set(
    data.select_dtypes(include=["number"]).columns))

TRAIN_MASK = (data['sample_type'] == 'TRAIN')
TEST_MASK = (data['sample_type'] == 'TEST')
OOT_MASK = (data['sample_type'] == 'OOT')

DEV_MASK = (data['competition_sample_type'] == 'TRAIN')

# Обучение модели

##  Утилиты

### FraudDataset

In [12]:
class FraudDataset(Dataset):
    """Датасет для последовательностей транзакций"""
    
    def __init__(self, sequences, targets):
        self.sequences = sequences  # список numpy массивов разной длины
        self.targets = targets      # список int
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.sequences[idx]), torch.FloatTensor([self.targets[idx]])


def collate_fn(batch):
    """Паддинг только для текущего батча"""
    sequences = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    
    lengths = torch.tensor([len(seq) for seq in sequences])
    sequences_padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    targets_tensor = torch.cat(targets, dim=0)
    
    return sequences_padded, targets_tensor, lengths

### FraudDataPreprocessor

In [13]:
class FraudDataPreprocessor:
    """Предобработка данных - создает последовательность для КАЖДОЙ транзакции (без паддинга)"""
    
    def __init__(self, cat_features, num_features, target_col='isFraud', 
                 max_seq_len=50):
        self.cat_features = cat_features
        self.num_features = num_features
        self.target_col = target_col
        self.max_seq_len = max_seq_len
        
        self.cat_encoders = {}
        self.num_scaler = StandardScaler()
        self.is_fitted = False
    
    def _encode_categorical(self, df, fit=False):
        if not self.cat_features:
            return pd.DataFrame(index=df.index)
        
        encoded_dfs = []
        for col in self.cat_features:
            if col not in df.columns:
                continue
            col_data = df[col].fillna('MISSING').astype(str)
            if fit:
                encoder = LabelEncoder()
                encoded = encoder.fit_transform(col_data)
                self.cat_encoders[col] = encoder
            else:
                encoder = self.cat_encoders[col]
                mapping = {cls: idx for idx, cls in enumerate(encoder.classes_)}
                encoded = col_data.map(mapping).fillna(-1).values.astype(np.int64)
            encoded_dfs.append(pd.Series(encoded, index=df.index, name=col))
        if encoded_dfs:
            return pd.concat(encoded_dfs, axis=1)
        return pd.DataFrame(index=df.index)
    
    def _scale_numerical(self, df, fit=False):
        available_num_features = [col for col in self.num_features if col in df.columns]
        if not available_num_features:
            return pd.DataFrame(index=df.index)
        num_data = df[available_num_features].fillna(0)
        if fit:
            scaled = self.num_scaler.fit_transform(num_data)
        else:
            scaled = self.num_scaler.transform(num_data)
        return pd.DataFrame(scaled, columns=available_num_features, index=df.index)
    
    def fit(self, df, client_id_col='card1'):
        print("  Кодирование категориальных признаков...")
        cat_encoded = self._encode_categorical(df, fit=True)
        print("  Масштабирование числовых признаков...")
        num_scaled = self._scale_numerical(df, fit=True)
        self.all_features = cat_encoded.columns.tolist() + num_scaled.columns.tolist()
        self.client_id_col = client_id_col
        self.is_fitted = True
        print(f"  Всего признаков: {len(self.all_features)}")
        return self
    
    def transform(self, df, sort_by_date_col='date_month'):
        """Создает последовательности БЕЗ паддинга для каждой транзакции"""
        if not self.is_fitted:
            raise ValueError("Препроцессор должен быть сначала обучен")
        
        print(f"  Трансформация {len(df)} строк...")
        
        cat_encoded = self._encode_categorical(df, fit=False)
        num_scaled = self._scale_numerical(df, fit=False)
        
        processed = pd.concat([cat_encoded, num_scaled], axis=1)
        
        if self.target_col in df.columns:
            processed[self.target_col] = df[self.target_col].values
        
        processed[self.client_id_col] = df[self.client_id_col].values
        processed['TransactionID'] = df['TransactionID'].values
        
        if sort_by_date_col and sort_by_date_col in df.columns:
            processed['_sort_date'] = df[sort_by_date_col].values
        else:
            processed['_sort_date'] = np.arange(len(df))
        
        processed = processed.sort_values([self.client_id_col, '_sort_date'])
        
        sequences = []
        targets = []
        transaction_ids = []
        
        grouped = processed.groupby(self.client_id_col)
        
        for client_id, group in tqdm(grouped, desc='    Создание последовательностей'):
            features = group[self.all_features].values
            targs = group[self.target_col].values
            trans_ids = group['TransactionID'].values
            
            for i in range(len(features)):
                start = max(0, i - self.max_seq_len + 1)
                seq = features[start:i+1]
                sequences.append(seq)
                targets.append(targs[i])
                transaction_ids.append(trans_ids[i])
        
        # Возвращаем БЕЗ паддинга
        return sequences, targets, transaction_ids

### RNNModel

In [14]:
class RNNModel(nn.Module):
    """RNN модель для детекции мошенничества - предсказание для каждой транзакции"""
    
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super(RNNModel, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        
        self.rnn = nn.RNN(
            input_dim, hidden_dim, num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x, lengths):
        """
        Args:
            x: (batch_size, max_seq_len, input_dim)
            lengths: (batch_size,)
        """
        batch_size = x.size(0)
        max_seq_len = x.size(1)
        
        # Сортируем по убыванию длины
        lengths, sort_idx = lengths.sort(descending=True)
        x = x[sort_idx]
        
        # Pack
        packed_x = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True)
        
        # RNN
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(x.device)
        packed_out, _ = self.rnn(packed_x, h0)
        
        # Unpack
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True, total_length=max_seq_len)
        
        # Возвращаем порядок
        _, unsort_idx = sort_idx.sort()
        out = out[unsort_idx]
        
        # Классификация для каждого шага
        logits = self.classifier(out).squeeze(-1)
        
        return logits

### RNNTrainer

In [15]:
class RNNTrainer:
    """Тренер для RNN модели - обучение на всех транзакциях"""
    
    def __init__(self, model, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.model = model.to(device)
        self.device = device
        self.history = defaultdict(list)
    
    def train_epoch(self, train_loader, criterion, optimizer):
        self.model.train()
        total_loss = 0
        all_preds = []
        all_targets = []
        
        for sequences, targets, lengths in tqdm(train_loader, desc='  Training', leave=False):
            sequences = sequences.to(self.device)
            targets = targets.to(self.device)
            lengths = lengths.to(self.device)
            
            logits = self.model(sequences, lengths)
            
            # Берем предсказание для последнего шага
            last_logits = logits[torch.arange(logits.size(0)), lengths - 1]
            
            loss = criterion(last_logits, targets)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            
            preds = torch.sigmoid(last_logits).detach().cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(targets.cpu().numpy())
        
        avg_loss = total_loss / len(train_loader)
        auc = roc_auc_score(all_targets, all_preds)
        
        return avg_loss, auc
    
    def evaluate(self, data_loader, criterion):
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for sequences, targets, lengths in tqdm(data_loader, desc='  Evaluating', leave=False):
                sequences = sequences.to(self.device)
                targets = targets.to(self.device)
                lengths = lengths.to(self.device)
                
                logits = self.model(sequences, lengths)
                last_logits = logits[torch.arange(logits.size(0)), lengths - 1]
                
                loss = criterion(last_logits, targets)
                total_loss += loss.item()
                
                preds = torch.sigmoid(last_logits).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(targets.cpu().numpy())
        
        avg_loss = total_loss / len(data_loader)
        auc = roc_auc_score(all_targets, all_preds)
        
        return avg_loss, auc
    
    def fit(self, train_loader, test_loader=None, oot_loader=None, 
            epochs=15, lr=0.001, weight_decay=1e-5):
        
        # Вычисляем веса для борьбы с дисбалансом
        pos_weight = None
        if train_loader is not None:
            all_targets = []
            for _, targets, _ in train_loader:
                all_targets.extend(targets.numpy())
            all_targets = np.array(all_targets)
            n_neg = (all_targets == 0).sum()
            n_pos = (all_targets == 1).sum()
            if n_pos > 0:
                pos_weight = torch.tensor([n_neg / n_pos]).to(self.device)
                print(f"  Pos weight: {pos_weight.item():.2f}")
        
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            
            train_loss, train_auc = self.train_epoch(train_loader, criterion, optimizer)
            
            self.history['train_loss'].append(train_loss)
            self.history['train_auc'].append(train_auc)
            
            print(f"  train - loss: {train_loss:.4f}, AUC: {train_auc:.4f}")
            
            if test_loader:
                test_loss, test_auc = self.evaluate(test_loader, criterion)
                self.history['test_loss'].append(test_loss)
                self.history['test_auc'].append(test_auc)
                print(f"  test  - loss: {test_loss:.4f}, AUC: {test_auc:.4f}")
                scheduler.step(test_loss)
            
            if oot_loader:
                oot_loss, oot_auc = self.evaluate(oot_loader, criterion)
                self.history['oot_loss'].append(oot_loss)
                self.history['oot_auc'].append(oot_auc)
                print(f"  oot   - loss: {oot_loss:.4f}, AUC: {oot_auc:.4f}")
            
            print("-" * 35)
        
        return self.history
    
    def predict(self, data_loader):
        self.model.eval()
        all_preds = []
        
        with torch.no_grad():
            for sequences, _, lengths in tqdm(data_loader, desc='Predicting'):
                sequences = sequences.to(self.device)
                lengths = lengths.to(self.device)
                logits = self.model(sequences, lengths)
                last_logits = logits[torch.arange(logits.size(0)), lengths - 1]
                preds = torch.sigmoid(last_logits).cpu().numpy()
                all_preds.extend(preds)
        
        return np.array(all_preds)

## Обучение

In [16]:
print(f"device: {torch.cuda.get_device_name(0)}")

device: Tesla T4


In [17]:
# Создание препроцессора
preprocessor = FraudDataPreprocessor(
    cat_features=CAT_FEATURES,
    num_features=NUM_FEATURES,
    target_col=TARGET,
    max_seq_len=20
)

print("\nОбучение препроцессора...")
preprocessor.fit(data[TRAIN_MASK], client_id_col='card1')

print("Трансформация данных...")
train_seq, train_targets, train_ids = preprocessor.transform(
    data[TRAIN_MASK], 
    sort_by_date_col=DATE_MONTH
)

test_seq, test_targets, test_ids = preprocessor.transform(
    data[TEST_MASK], 
    sort_by_date_col=DATE_MONTH
)

oot_seq, oot_targets, oot_ids = preprocessor.transform(
    data[OOT_MASK], 
    sort_by_date_col=DATE_MONTH
)

print(f"\nTrain: {len(train_seq)} транзакций")
print(f"Test: {len(test_seq)} транзакций")
print(f"OOT: {len(oot_seq)} транзакций")

print(f"\nTRAIN target=1: {np.sum(train_targets)}/{len(train_targets)} ({np.mean(train_targets)*100:.4f}%)")
print(f"TEST target=1: {np.sum(test_targets)}/{len(test_targets)} ({np.mean(test_targets)*100:.4f}%)")
print(f"OOT target=1: {np.sum(oot_targets)}/{len(oot_targets)} ({np.mean(oot_targets)*100:.4f}%)")


# Создание датасетов
train_dataset = FraudDataset(train_seq, train_targets)
test_dataset = FraudDataset(test_seq, test_targets)
oot_dataset = FraudDataset(oot_seq, oot_targets)

# Создание DataLoader'ов
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
oot_loader = DataLoader(oot_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


Обучение препроцессора...
  Кодирование категориальных признаков...
  Масштабирование числовых признаков...
  Всего признаков: 130
Трансформация данных...
  Трансформация 350849 строк...


    Создание последовательностей: 100%|██████████| 11740/11740 [00:09<00:00, 1294.37it/s]


  Трансформация 150365 строк...


    Создание последовательностей: 100%|██████████| 8924/8924 [00:05<00:00, 1559.06it/s]


  Трансформация 89326 строк...


    Создание последовательностей: 100%|██████████| 6334/6334 [00:05<00:00, 1239.35it/s]



Train: 350849 транзакций
Test: 150365 транзакций
OOT: 89326 транзакций

TRAIN target=1: 12276.0/350849 (3.4989%)
TEST target=1: 5273.0/150365 (3.5068%)
OOT target=1: 3114.0/89326 (3.4861%)


In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

# input_dim - берем из первой последовательности
input_dim = train_seq[0].shape[1]  # (seq_len, features)
print(f"\nРазмер входных признаков: {input_dim}")

model = RNNModel(
    input_dim=input_dim,
    hidden_dim=64,
    num_layers=1,
    dropout=0.3
)

trainer = RNNTrainer(model, device=device)
print("\nНачало обучения...")

history = trainer.fit(
    train_loader=train_loader,
    test_loader=test_loader,
    oot_loader=oot_loader,
    epochs=10,
    lr=0.001,
    weight_decay=1e-5
)

torch.save(model.state_dict(), r'/content/drive/MyDrive/colab/year_project/models/rnn_fraud_model.pth')
print("\nМодель сохранена как 'rnn_fraud_model.pth'")

Используется устройство: cuda

Размер входных признаков: 130

Начало обучения...
  Pos weight: 27.58

Epoch 1/10


  train - loss: 1.0975, AUC: 0.7986


  test  - loss: 1.0361, AUC: 0.8338


  oot   - loss: 1.0755, AUC: 0.8212
-----------------------------------

Epoch 2/10


  train - loss: 1.0350, AUC: 0.8273


...

  test  - loss: 1.2857, AUC: 0.8617


  oot   - loss: 1.3482, AUC: 0.8415
-----------------------------------

Epoch 10/10


  train - loss: 0.9748, AUC: 0.8633


  test  - loss: 1.0122, AUC: 0.8662


  oot   - loss: 1.0900, AUC: 0.8445
-----------------------------------

Модель сохранена как 'rnn_fraud_model.pth'


# Сохранение предсказаний

In [19]:
print("\nПолучение предсказаний...")

# Создаем DataLoader'ы без shuffle для предсказаний
train_loader_eval = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader_eval = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
oot_loader_eval = DataLoader(oot_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# Предсказания для каждой выборки
train_preds = trainer.predict(train_loader_eval)
test_preds = trainer.predict(test_loader_eval)
oot_preds = trainer.predict(oot_loader_eval)

# Собираем TransactionID (теперь порядок совпадает с preds)
train_ids = data[TRAIN_MASK]['TransactionID'].values
test_ids = data[TEST_MASK]['TransactionID'].values
oot_ids = data[OOT_MASK]['TransactionID'].values

# Собираем targets
train_targs = np.array(train_targets)
test_targs = np.array(test_targets)
oot_targs = np.array(oot_targets)

# Проверка размеров
print(f"Train: preds={len(train_preds)}, ids={len(train_ids)}, targets={len(train_targs)}")
print(f"Test: preds={len(test_preds)}, ids={len(test_ids)}, targets={len(test_targs)}")
print(f"OOT: preds={len(oot_preds)}, ids={len(oot_ids)}, targets={len(oot_targs)}")


Получение предсказаний...


Predicting: 100%|██████████| 1396/1396 [00:05<00:00, 250.85it/s]


Train: preds=350849, ids=350849, targets=350849
Test: preds=150365, ids=150365, targets=150365
OOT: preds=89326, ids=89326, targets=89326


In [20]:
# Итоговый DataFrame
all_results = pd.DataFrame({
    'TransactionID': np.concatenate([train_ids, test_ids, oot_ids]),
    'target': np.concatenate([train_targs, test_targs, oot_targs]),
    'prediction': np.concatenate([train_preds, test_preds, oot_preds]),
    'sample_type': ['TRAIN'] * len(train_preds) + ['TEST'] * len(test_preds) + ['OOT'] * len(oot_preds)
})

print(f"\nВсего транзакций: {len(all_results)}")
print(all_results.head())

# Сохраняем в Drive
all_results.to_parquet('/content/drive/MyDrive/colab/year_project/models/all_predictions.pqt', index=False)
print("\nСохранено в '/content/drive/MyDrive/colab/year_project/models/all_predictions.pqt'")


Всего транзакций: 590540
   TransactionID  target  prediction sample_type
0        3230924    0.00        0.57       TRAIN
1        3023634    0.00        0.03       TRAIN
2        3210739    0.00        0.01       TRAIN
3        3020767    0.00        0.39       TRAIN
4        3028973    0.00        0.02       TRAIN

Сохранено в '/content/drive/MyDrive/colab/year_project/models/all_predictions.pqt'


In [23]:
# Метрики
for sample_type in ['TRAIN', 'TEST', 'OOT']:
    subset = all_results[all_results['sample_type'] == sample_type]
    if len(subset) > 0 and subset['target'].sum() > 0:
        auc = roc_auc_score(subset['target'], subset['prediction'])
        print(f"  {sample_type}: AUC = {auc:.4f}")

  TRAIN: AUC = 0.8844
  TEST: AUC = 0.8662
  OOT: AUC = 0.8445


In [24]:
all_results.shape

(590540, 4)